# Feature Engineering

Build the modeling table from `health_and_wellness_no_outliers.csv`.

**Target: `log_price_thb` = `log1p(Price)`** — *price recommendation*: predict a fair listing
price from catalog attributes (section, shop location, name text, review signal).

Anything that is a function of `Price` (e.g. price-vs-section, price rank) is **leakage**
against this target and is intentionally **not** engineered. `Total Sold` (the previous
notebook's candidate target) is also dropped — it is a post-launch signal, mostly missing,
and not a price predictor.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "dataset" / "health_and_wellness_no_outliers.csv"
OUTPUT_PATH = PROJECT_ROOT / "dataset" / "health_and_wellness_feature_engineered.csv"

raw_df = pd.read_csv(DATA_PATH)
raw_df.head()

,Id,Section,Name,Price,Total Sold,Total Reviews,Shop Location
0,4799179886,Acne Care,DHC Vitamin B-Mix วิตามินบีรวม (สำหรับ 20 วัน),75.0,NaN,NaN,Pathum Thani
1,4795171651,Acne Care,ซิงค์ Vistra Zinc วิสทร้า ซิงค์ 15 มก. ขนาด 20...,79.0,NaN,NaN,Chiang Mai
2,4790242641,Acne Care,Blackmores แบลคมอร์ส Bio Zinc A Chelate (90 T...,224.0,NaN,NaN,Surin
3,4789940606,Acne Care,1แถม1 กลูต้าวิตมี กลูต้าส้มเลือด Gluta With Me...,290.0,NaN,NaN,Udon Thani
4,4787161067,Acne Care,🎌 DHC Vitamin B-Mix Persistent วิตามินบีรวม แบ...,149.0,NaN,NaN,Bangkok


## Column Explanations

**Target**

| Column | Explanation |
|---|---|
| `log_price_thb` | The prediction target — `log(1 + Price)`, the log-transformed listing price in Thai baht. The log tames the heavy right skew (median ฿299, max ฿5,600) seen in EDA. |

**Features**

| Column | Explanation |
|---|---|
| `log_total_reviews` | `log(1 + Total Reviews)` — a sparse demand/popularity signal. Kept per the EDA plan; ~40% of rows are `0`. |
| `has_reviews` | Binary `1` if `Total Reviews > 0` else `0`. Captures the presence of any market traction. |
| `name_length` | Character count of the product `Name`. Proxy for how detailed / keyword-heavy the title is. |
| `name_word_count` | Whitespace-split word count of `Name`. Another measure of title detail. |
| `section_*` | One-hot of `Section` (e.g. `section_Acne_Care`). Each product gets `1` for its own section. |
| `shop_location_*` | One-hot of `Shop Location` (e.g. `shop_location_Bangkok`). Each product gets `1` for its location. |

The one-hot columns are created for every category in the raw dataset, so the final table
includes all 30 product sections and all shop locations.

**Deliberately excluded** (leakage / not a price predictor):
`price_vs_section_mean`, `price_ratio_to_section_mean`, `price_rank_in_section` — all
functions of `Price`, so they would leak the target. `Total Sold` — the previous candidate
target; post-launch and ~46% missing.

## Build Features

In [3]:
import re
import sys
from pathlib import Path

# Add project root to path so the notebook can import src.regions.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.regions import PROVINCE_TO_REGION, KNOWN_REGIONS


def clean_category_value(value):
    """Create readable, stable dummy-column suffixes from category values."""
    value = "Unknown" if pd.isna(value) else str(value).strip()
    value = re.sub(r"\W+", "_", value, flags=re.UNICODE).strip("_")
    return value or "Unknown"


def map_to_region(cleaned_province):
    """Return the region for a cleaned province name; unknown provinces fall back."""
    return PROVINCE_TO_REGION.get(cleaned_province, "unknown")


df = raw_df.copy()

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Total Reviews"] = pd.to_numeric(df["Total Reviews"], errors="coerce").fillna(0)
df["Section"] = df["Section"].fillna("Unknown")
df["Shop Location"] = df["Shop Location"].fillna("Unknown")
df["Name"] = df["Name"].fillna("")

features = pd.DataFrame(index=df.index)
features["log_price_thb"] = np.log1p(df["Price"])          # target: price recommendation
features["log_total_reviews"] = np.log1p(df["Total Reviews"])
features["has_reviews"] = (df["Total Reviews"] > 0).astype(int)
features["name_length"] = df["Name"].str.len()
features["name_word_count"] = df["Name"].str.split().str.len().fillna(0).astype(int)

# ponytail: price_vs_section_mean / price_ratio / price_rank are functions of Price —
# leakage against the price target, so they are intentionally NOT engineered here.
# Total Sold (the prior candidate target) is dropped too: post-launch and ~46% missing.

categorical_df = pd.DataFrame(
    {
        "section": df["Section"].map(clean_category_value),
        "shop_region": df["Shop Location"].map(lambda x: map_to_region(clean_category_value(x))),
    }
)

dummy_features = pd.get_dummies(
    categorical_df,
    columns=["section", "shop_region"],
    prefix=["section", "shop_region"],
    dtype=int,
)

feature_df = pd.concat([features, dummy_features], axis=1)
feature_df = feature_df.replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.fillna(0)

feature_df.head()

,log_price_thb,log_total_reviews,has_reviews,name_length,name_word_count,section_Acne_Care,section_Appetite_Suppressant,section_Beauty_Supplements_Value_Sets,section_Bone_Joint_Support,section_Brain_Memory,...,section_Women_s_Health,shop_region_bangkok_metro,shop_region_central,shop_region_eastern,shop_region_northeastern,shop_region_northern,shop_region_overseas,shop_region_southern,shop_region_unknown,shop_region_western
0,4.330733,0.0,0,46,7,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,4.382027,0.0,0,53,10,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,5.416100,0.0,0,118,18,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,5.673323,0.0,0,72,8,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,5.010635,0.0,0,119,12,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


## Save Final Table

In [4]:
feature_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved feature table to: {OUTPUT_PATH}")
print(f"Rows: {feature_df.shape[0]:,}")
print(f"Columns: {feature_df.shape[1]:,}")

Saved feature table to: /Users/akarapongmba/Documents/GitHub/hs-practical-machine-learning-final-project/dataset/health_and_wellness_feature_engineered.csv
Rows: 2,610
Columns: 44


In [5]:
feature_df.filter(
    regex=r"^(log_price_thb|log_total_reviews|has_reviews|name_|section_|shop_location_)"
).head()

,log_price_thb,log_total_reviews,has_reviews,name_length,name_word_count,section_Acne_Care,section_Appetite_Suppressant,section_Beauty_Supplements_Value_Sets,section_Bone_Joint_Support,section_Brain_Memory,...,section_Pregnancy_Care,section_Protein,section_Sexual_Health_Vitamins,section_Skin_Nourishment,section_Slimming_Beverages,section_Stress_Sleep_and_Anxiety,section_Weight_Management_Value_Sets,section_Well_Being_Gifts_Value_Sets,section_Whitening,section_Women_s_Health
0,4.330733,0.0,0,46,7,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,4.382027,0.0,0,53,10,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5.416100,0.0,0,118,18,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,5.673323,0.0,0,72,8,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5.010635,0.0,0,119,12,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
